# Event Hub Consumer — Wikipedia Recent Changes

Reads events from the personal Event Hub `ivanrazumovskyi_evh` via the Kafka-compatible
endpoint (no extra Maven library required — Spark's built-in Kafka source works directly
against Event Hub's Kafka API). Bronze layer stores the raw event JSON as-is, plus Kafka
delivery metadata (topic/partition/offset) and standard ingestion metadata — full parsing
into structured columns is deferred to Silver, so Bronze stays resilient to upstream
schema changes on the Wikimedia side.

## 1. Configuration

In [0]:
%run ./lab3_00_config

In [0]:
pipeline_name = "wikipedia_edits_stream"

checkpoint_path = f"{storage_root}/checkpoints/{pipeline_name}/"
target_table = f"{catalog}.{bronze_schema}.{pipeline_name}"

print("Event Hub namespace:", namespace_name)
print("Event Hub name:", eh_name)
print("Checkpoint path:", checkpoint_path)
print("Target table:", target_table)

## 2. Load connection string and build Kafka-compatible options

In [0]:
conn_string = dbutils.secrets.get(scope=secret_scope, key=eventhub_secret_key)
print("Connection string loaded, length:", len(conn_string))

bootstrap_servers = f"{namespace_name}.servicebus.windows.net:9093"

sasl_config = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
    'username="$ConnectionString" '
    f'password="{conn_string}";'
)

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": eh_name,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": sasl_config,
    "startingOffsets": "earliest",
}

print("Bootstrap servers:", bootstrap_servers)

## 3. Read from Event Hub via Kafka source

In [0]:
df_raw_stream = spark.readStream.format("kafka").options(**kafka_options).load()
df_raw_stream.printSchema()

## 4. Decode payload, keep raw JSON + Kafka delivery metadata

In [0]:
from pyspark.sql import functions as F

df_decoded = df_raw_stream.select(
    F.col("value").cast("string").alias("event_json"),
    F.col("topic").alias("kafka_topic"),
    F.col("partition").alias("kafka_partition"),
    F.col("offset").alias("kafka_offset"),
    F.col("timestamp").alias("kafka_enqueued_at"),
)

## 5. Add standard ingestion metadata

In [0]:
df_bronze_stream = (
    df_decoded
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_load_date", F.current_date())
)

## 6. Write to the Bronze Delta table

In [0]:
query = (
    df_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

row_count = spark.table(target_table).count()
print("Row count after initial load:", row_count)

## 7. Sanity check — inspect a few rows

In [0]:
display(spark.table(target_table).limit(5))

## 8. Idempotency test — restart with the same checkpoint

In [0]:
before_count = spark.table(target_table).count()
print(f"Row count BEFORE re-run: {before_count}")

In [0]:
query = (
    df_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
after_count = spark.table(target_table).count()
print(f"Row count AFTER re-run: {after_count}")

if after_count == before_count:
    print("✅ IDEMPOTENCY TEST PASSED — Kafka offsets in the checkpoint prevented re-reading already-consumed events")
else:
    print(f"❌ IDEMPOTENCY TEST FAILED — row count changed by {after_count - before_count}")

## 9. UDF decision

No UDFs used here — casting `value` to string, selecting Kafka metadata, and adding
timestamps are all covered by built-in Spark functions (`cast`, `current_timestamp`,
`current_date`), which are faster than UDFs since they avoid Python/JVM serialization
per row.

## Scheduled Job

This consumer notebook runs as part of a two-task Databricks Job,
`ivanrazumovskyi-bronze-ingestion-lab3`, on the shared all-purpose cluster:

1. **producer_task** — sends ~50 filtered Wikipedia edit events to the personal
   Event Hub (`ivanrazumovskyi_evh`)
2. **consumer_task** *(this notebook)* — depends on `producer_task`; reads new
   events via the Kafka-compatible endpoint and appends them to Bronze


Job link: [ivanrazumovskyi-bronze-ingestion-lab3](https://adb-7405604503619901.1.azuredatabricks.net/jobs/678122748410936?o=7405604503619901)